In [19]:
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import h5py
import os,json
from pathlib import Path
from pprint import pprint
import dask.dataframe as dd

In [22]:
# import geopandas as gpd
# gpd.show_versions()  # 看 fiona/pyogrio/gdal 是否都有版本号

In [23]:
# import sys, geopandas as gpd, shapely
# print("PYTHON:", sys.executable)
# print("GEOPANDAS:", gpd.__file__)

### 提取参数
- 掩膜切片
- 参数提取
- 存为dataframe格式

In [30]:
def read_atl08(fname, bbox=None, poly=None):
    group = ['/gt1l','/gt1r','/gt2l','/gt2r','/gt3l','/gt3r']
    all_beams = []

    with h5py.File(fname, 'r') as fi:
        for g in group:
            ls = f"{g}/land_segments"
            if ls not in fi:
                continue
            grp = fi[ls]

            lat = grp["latitude"][:]
            lon = grp["longitude"][:]

            if bbox is not None:
                lonmin, lonmax, latmin, latmax = bbox
                m = (lon >= lonmin) & (lon <= lonmax) & (lat >= latmin) & (lat <= latmax)
                if not np.any(m):
                    continue
                lat, lon = lat[m], lon[m]

            def take(path, is2d=False):
                arr = grp[path][:]
                return arr[m, :] if (bbox is not None and is2d) else (arr[m] if bbox is not None else arr)

            dem_h         = take("dem_h")
            h_te_interp   = take("terrain/h_te_interp")
            h_te_mean     = take("terrain/h_te_mean")
            h_te_best_fit = take("terrain/h_te_best_fit")
            terrain_flg   = take("terrain_flg")
            urban_flag    = take("urban_flag")
            cloud_flag_atm= take("cloud_flag_atm")
            terrain_slope = take("terrain/terrain_slope")
            n_te_photons  = take("terrain/n_te_photons")

            df = pd.DataFrame({
                "beam": g[1:], "lat": lat, "lon": lon,
                "dem_h": dem_h, "h_te_interp": h_te_interp, "h_te_mean": h_te_mean,
                "h_te_best_fit": h_te_best_fit, "terrain_flg": terrain_flg,
                "urban_flag": urban_flag, "cloud_flag_atm": cloud_flag_atm,
                "terrain_slope": terrain_slope, "n_te_photons": n_te_photons
            })
            all_beams.append(df)
            
    if poly is not None and len(all_beams) > 0:
        df_all = pd.concat(all_beams, ignore_index=True)
        import geopandas as gpd
        from shapely.geometry import Point
        # 将 lon/lat 转成点要素（同样是 EPSG:4326）
        pts = gpd.GeoSeries([Point(xy) for xy in zip(df_all["lon"], df_all["lat"])], crs="EPSG:4326")
        poly_geom = poly.union_all()
        # within：点在多边形内部（严格意义）；如果想包含边界可用 .covers()   
        in_poly = pts.within(poly_geom)
        # 用多边形掩膜过滤
        df_all = df_all.loc[in_poly].reset_index(drop=True)
        return df_all


    return pd.concat(all_beams, ignore_index=True) if all_beams else pd.DataFrame()

### 打印树结构
- 看所有变量位置

In [25]:
# fname = r"/Users/a1-6/Self_program/underground_water/ATL08_007-20251002_204153/ATL08_20220102113835_01711406_007_01.h5"

# with h5py.File(fname, 'r') as f:
#     def print_tree(name, obj):
#         print(name)
#     f.visititems(print_tree)

### 测试read_at108函数

In [26]:
fname = r"/Users/a1-6/Self_program/underground_water/ATL08_007-20251002_204153/ATL08_20220102113835_01711406_007_01.h5"

df = read_atl08(fname)

print("数据总行数:", len(df))
print("列名:", df.columns.tolist())
print("前几行:")
print(df.head())

# 简单统计一下关键字段
print("\nlat 范围:", df["lat"].min(), "→", df["lat"].max())
print("lon 范围:", df["lon"].min(), "→", df["lon"].max())
print("高程范围:", df["h_te_best_fit"].min(), "→", df["h_te_best_fit"].max())

# 按束分组统计数量
print("\n每束点数统计:")
print(df["beam"].value_counts())

数据总行数: 170237
列名: ['beam', 'lat', 'lon', 'dem_h', 'h_te_interp', 'h_te_mean', 'h_te_best_fit', 'terrain_flg', 'urban_flag', 'cloud_flag_atm', 'terrain_slope', 'n_te_photons']
前几行:
   beam        lat         lon       dem_h  h_te_interp   h_te_mean  \
0  gt1l  59.496704  120.130447  367.754669   365.817505  366.636871   
1  gt1l  59.495811  120.130257  376.254944   375.866669  375.207092   
2  gt1l  59.494919  120.130074  386.441803   383.495880  383.506744   
3  gt1l  59.494022  120.129898  395.323608   391.877075  390.289825   
4  gt1l  59.493130  120.129723  403.160248   399.442322  399.757782   

   h_te_best_fit  terrain_flg  urban_flag  cloud_flag_atm  terrain_slope  \
0     366.457367            1           0               1       0.088511   
1     375.665253            0           0               1       0.076993   
2     383.434631            1           0               1       0.085131   
3     391.860168            1           0               1       0.087166   
4     399.526

### 生成位掩码 bitmask
- `masks` 字典：每个点单独条件是否通过（方便统计单个条件的通过率）。
- `bitmask` 数组：每个点一个整数，记录它通过了哪些条件。
    - 例如 bitmask = `0b010101101` → 表示通过了 a, c, e, g, i。


In [27]:
def build_masks(df, sigma_a_b=25.0, thr_cloud=3, thr_nphot=20, thr_slope=0.01):
    """返回 a..i 的布尔列，以及 bitmask（int32）"""
    # a: |dem_h - h_te_interp| <= 3σ（σ取25 m）
    a = (np.abs(df.dem_h - df.h_te_interp) <= 3 * sigma_a_b)

    # b: |h_te_mean - h_te_interp| <= 3σ（σ取25 m）
    b = (np.abs(df.h_te_mean - df.h_te_interp) <= 3 * sigma_a_b)

    # c: terrain_flg == 0
    c = (df.terrain_flg == 0)

    # d: urban_flag == 0
    d = (df.urban_flag == 0)

    # e: cloud_flag_atm < 3
    e = (df.cloud_flag_atm < thr_cloud)

    # f: terrain_slope < 0.01
    f = (df.terrain_slope < thr_slope)

    # g: n_te_photons > 20
    g = (df.n_te_photons > thr_nphot)

    # h: |dem_h - h_te_interp| <= 3
    h = (np.abs(df.dem_h - df.h_te_interp) <= 3.0)

    # i: |dem_h - h_te_best_fit| <= 3
    i = (np.abs(df.dem_h - df.h_te_best_fit) <= 3.0)

    masks = {'a':a,'b':b,'c':c,'d':d,'e':e,'f':f,'g':g,'h':h,'i':i}

    bitmask = np.zeros(len(df), dtype=np.int32)
    for bit,key in enumerate(['a','b','c','d','e','f','g','h','i']):
        bitmask |= (masks[key].values.astype(np.int32) << bit)

    return masks, bitmask

### 测试
- 输出各个约束的通过率

In [28]:
# fname = r'/Users/a1-6/Self_program/underground_water/ATL08_007-20251002_204153/ATL08_20220102113835_01711406_007_01.h5'
# bbox = (115.5, 116.8, 28.2, 29.9) 
# df = read_atl08(fname, bbox=bbox)
# masks, bitmask = build_masks(df)

# for k,v in masks.items():
#     print(f"{k}: {v.sum()} / {len(v)}  ({v.mean():.2%})")

# need = ['a','b','c','d','e']
# order = ['a','b','c','d','e','f','g','h','i']
# need_bits = sum(1 << order.index(k) for k in need)
# ok = (bitmask & need_bits) == need_bits

# print(f"法则4 通过: {ok.sum()} / {len(ok)}")
# df_rule4 = df[ok]  
# df_rule4.to_csv("controlpoints_rule4.csv", index=False)

In [41]:

files = [
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_175955/ATL08_20220102113835_01711406_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_175955/ATL08_20220111230607_03161402_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_175955/ATL08_20220115225748_03771402_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_175955/ATL08_20220119224927_04381402_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221103210552_06741706_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221109084141_07581702_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221113083320_08191702_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221117082503_08801702_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221124195834_09941706_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221128195015_10551706_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221202194153_11161706_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221212070923_12611702_007_01.h5',
    r'/Users/a1-6/Self_program/underground_water/Jan_Nov_Dec/ATL08_007-20251005_180052/ATL08_20221216070101_13221702_007_01.h5',
    
]
# bbox = (115.5, 116.8, 28.2, 29.9) 

#用shp文件进行掩膜
poly = gpd.read_file("/Users/a1-6/Self_program/underground_water/鄱阳湖最大边界/Poyang_Lakes1.shp")
if poly.crs is None:
    raise ValueError("Shapefile 没有坐标系（CRS），请先在 GIS 里定义或确认")
poly = poly.to_crs(4326)  # EPSG:4326 = WGS84 经纬度

rules = {
    "Rule1": list("ab"),
    "Rule2": list("abc"),
    "Rule3": list("abcd"),
    "Rule4": list("abcde"),
    "Rule5": list("abcdef"),
    "Rule6": list("abcdefg"),
    "Rule7": list("hi"),
    "Rule8": list("abcdefghi"),
}
order = list("abcdefghi") 
bit_of = {k: i for i, k in enumerate(order)}

out_dir = Path("controlpoints_out")
out_dir.mkdir(parents=True, exist_ok=True)

all_summary = []
no_data_files = [] 
for fname in files:  
    stem = Path(fname).stem  
    df = read_atl08(fname, poly=poly)
    if df is None or df.empty or ("beam" not in df.columns):
        no_data_files.append(Path(fname).name) 
        continue
    df = df.replace([np.inf, -np.inf], np.nan)



    print(f"\n=== 处理: {stem} ===")
    print(f"掩膜后 DataFrame 总点数：{len(df):,}")
    print("各束点数统计：")
    print(df["beam"].value_counts())


    masks, bitmask = build_masks(df)

    summary = [] 
    for rule_name, need in rules.items():
        need_bits = sum(1 << bit_of[k] for k in need) 
        ok = (bitmask & need_bits) == need_bits

        cols_full = ["beam","lon","lat","h_te_best_fit","h_te_interp","h_te_mean",
                     "dem_h","terrain_flg","urban_flag","cloud_flag_atm",
                     "terrain_slope","n_te_photons"]

        df.loc[ok, cols_full].copy().to_csv(out_dir / f"{stem}_{rule_name}_full.csv", index=False)

        n_pass = int(ok.sum())
        rate = float(ok.mean()) if len(ok) else 0.0
        summary.append({
            "rule": rule_name,
            "need": "+".join(need),
            "N_pass": n_pass,
            "rate": rate
        })
    for row in summary:
        row["file"] = stem
        all_summary.append(row)
 
    pd.DataFrame(summary).to_csv(out_dir / f"{stem}_rule_summary.csv", index=False)
    print(pd.DataFrame(summary))

if no_data_files:
    print("\n⚠️ 以下文件在掩膜区域内没有数据：")
    for f in no_data_files:
        print("  -", f)

df_all_rules = pd.DataFrame(all_summary)

df_summary_total = (
    df_all_rules.groupby("rule", as_index=False)
    .agg({"N_pass": "sum"})
    .rename(columns={"N_pass": "Total_N_pass"})
)

total_points = 0
for fname in files:
    df_tmp = read_atl08(fname, poly=poly)
    if df_tmp is not None and not df_tmp.empty:
        total_points += len(df_tmp)

df_summary_total["Global_reject_rate"] = 1 - (df_summary_total["Total_N_pass"] / total_points)

df_summary_total.to_csv(out_dir / "rules_total_summary.csv", index=False)
print("\n=== 汇总 ===")
print(df_summary_total)


=== 处理: ATL08_20220102113835_01711406_007_01 ===
掩膜后 DataFrame 总点数：145
各束点数统计：
beam
gt1l    56
gt2r    32
gt2l    29
gt1r    28
Name: count, dtype: int64
    rule               need  N_pass      rate  \
0  Rule1                a+b     143  0.986207   
1  Rule2              a+b+c      43  0.296552   
2  Rule3            a+b+c+d      43  0.296552   
3  Rule4          a+b+c+d+e      43  0.296552   
4  Rule5        a+b+c+d+e+f      41  0.282759   
5  Rule6      a+b+c+d+e+f+g      41  0.282759   
6  Rule7                h+i      87  0.600000   
7  Rule8  a+b+c+d+e+f+g+h+i      41  0.282759   

                                   file  
0  ATL08_20220102113835_01711406_007_01  
1  ATL08_20220102113835_01711406_007_01  
2  ATL08_20220102113835_01711406_007_01  
3  ATL08_20220102113835_01711406_007_01  
4  ATL08_20220102113835_01711406_007_01  
5  ATL08_20220102113835_01711406_007_01  
6  ATL08_20220102113835_01711406_007_01  
7  ATL08_20220102113835_01711406_007_01  

=== 处理: ATL08_2022011123